# XGBoost Regression


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\moham\Desktop\Data Science\BeCode\becode_projects\immo-eliza\data\processed\listings_with_distance_clean.csv')


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23169 entries, 0 to 23168
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   region                  23169 non-null  str    
 2   zip_code                23169 non-null  int64  
 3   property_type           23169 non-null  str    
 4   subtype                 23169 non-null  str    
 5   price_eur               23169 non-null  float64
 6   type_of_sale            23169 non-null  str    
 7   num_rooms               23169 non-null  int64  
 8   living_area_m2          23169 non-null  float64
 9   fully_equipped_kitchen  23169 non-null  int64  
 10  furnished               23169 non-null  int64  
 11  terrace                 23169 non-null  int64  
 12  terrace_area_m2         15626 non-null  float64
 13  garden                  23169 non-null  int64  
 14  garden_area_m2          15292 non-null  float64
 

In [3]:
#splitting zip code
#use first two digits to get specific info about municipalities/provinces

df['zip_code'] = df['zip_code'] // 100 

In [4]:
#define y target variable
y = df['price_eur']

#define X variable by dropping irrelevant columns (pure noise) and target
X = df.drop(columns=['locality', 'subtype', 'type_of_sale', 'price_eur'])

In [5]:
# 1. Define your purely categorical columns
cat_cols = ['region', 'property_type', 'zip_code']

# 2. Force Pandas to treat them as Categories
for col in cat_cols:
    X[col] = X[col].astype('category')

In [6]:
df.info()
X

<class 'pandas.DataFrame'>
RangeIndex: 23169 entries, 0 to 23168
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   region                  23169 non-null  str    
 2   zip_code                23169 non-null  int64  
 3   property_type           23169 non-null  str    
 4   subtype                 23169 non-null  str    
 5   price_eur               23169 non-null  float64
 6   type_of_sale            23169 non-null  str    
 7   num_rooms               23169 non-null  int64  
 8   living_area_m2          23169 non-null  float64
 9   fully_equipped_kitchen  23169 non-null  int64  
 10  furnished               23169 non-null  int64  
 11  terrace                 23169 non-null  int64  
 12  terrace_area_m2         15626 non-null  float64
 13  garden                  23169 non-null  int64  
 14  garden_area_m2          15292 non-null  float64
 

,region,zip_code,property_type,num_rooms,living_area_m2,fully_equipped_kitchen,furnished,terrace,terrace_area_m2,garden,garden_area_m2,land_surface_m2,num_facades,swimming_pool,state_of_building,num_bathrooms,dist_train_km,dist_bus_km
0,Wallonia,71,House,5,450.0,1,0,1,NaN,1,NaN,12700.0,4.0,0,NaN,3.0,5.000,0.514
1,Flanders,31,House,4,171.0,0,0,1,NaN,1,NaN,400.0,3.0,0,Normal,2.0,1.500,0.750
2,Flanders,23,House,2,128.0,0,0,0,0.0,1,NaN,189.0,3.0,0,Normal,1.0,3.700,0.591
3,Wallonia,44,House,2,95.0,0,0,0,0.0,0,0.0,31.0,3.0,0,To renovate,1.0,4.300,0.321
4,Wallonia,14,House,4,175.0,0,0,1,30.0,1,200.0,NaN,2.0,0,Excellent,NaN,1.800,1.300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23164,Brussels,11,House,5,489.0,1,0,1,45.0,1,1100.0,1400.0,4.0,0,Excellent,5.0,0.847,0.137
23165,Flanders,94,House,4,295.0,0,0,1,50.0,1,1500.0,NaN,NaN,0,Normal,2.0,3.100,0.724
23166,Flanders,36,House,3,165.0,0,0,1,NaN,1,306.0,535.0,3.0,0,To be renovated,NaN,NaN,0.078
23167,Flanders,26,Apartment,2,78.0,0,0,1,NaN,0,0.0,0.0,2.0,0,To be renovated,NaN,0.760,1.600


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(18535, 18)
(4634, 18)
(18535,)
(4634,)


In [8]:
numeric_features = ['num_rooms', 
                      'living_area_m2', 
                      'terrace_area_m2', 
                      'garden_area_m2', 
                      'land_surface_m2', 
                      'num_facades', 
                      'num_bathrooms',  
                      'dist_train_km', 
                      'dist_bus_km']


nominal_features = ['region', 
                        'zip_code', 
                        'property_type', 
                        'fully_equipped_kitchen', 
                        'furnished', 
                        'terrace', 
                        'garden', 
                        'swimming_pool']


ordinal_features = ['state_of_building']

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

In [10]:
#create pipeline for ordinal state_of_building 
#to impute NANs with most_frequent value 
#and rank the states from lowest to highest
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories= [['To demolish', 
                                            'To restore', 
                                            'To be renovated', 
                                            'To renovate', 
                                            'Under construction',
                                            'Normal', 
                                            'Excellent', 
                                            'Fully renovated', 
                                            'New']],
                                handle_unknown='use_encoded_value',
                                unknown_value=-1))
                                
])

In [11]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('ord', ordinal_transformer, ordinal_features)
    ],
    remainder='passthrough'
)

preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fea

Model instantiation w/ native model category recognition

In [12]:
import xgboost as xgb

# 3. CRITICAL: When you build the model, you MUST turn the feature on:
xgbr_model_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        objective='reg:squarederror', # Tell it to predict continuous numbers
        tree_method='hist',           # REQUIRED to use the native category trick
        enable_categorical=True,      # Turns on the superpower
        random_state=42
    ))])


### randomizedsearchcv
to apply to xgboost


In [13]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'regressor__n_estimators': [100, 300, 500],
    'regressor__learning_rate': [0.01, 0.05, 0.1],
    'regressor__max_depth': [4, 6, 8],          # Keep it relatively shallow
    'regressor__subsample': [0.7, 0.8, 1.0],    # The 80% data trick to prevent overfitting
    'regressor__colsample_bytree': [0.7, 0.8, 1.0] # Forces trees to use different columns
}

XGBR_grid_search = RandomizedSearchCV(
    estimator=xgbr_model_pipe, 
    param_distributions=param_grid, 
    cv=5, 
    scoring='neg_root_mean_squared_error', # REGRESSION SCORING
    n_jobs=2,
    n_iter=10,
    random_state=42)

XGBR_grid_search.fit(X_train, y_train)

print(f"Best Score: {XGBR_grid_search.best_score_}")
print(f"Best Params: {XGBR_grid_search.best_params_}")

#he grid search automatically saves the winning model inside an attribute called best_estimator_
XGBR_best_model = XGBR_grid_search.best_estimator_
print(f"Best Model Train Score {XGBR_best_model.score(X_train, y_train)}")
print(f"Best Model Test Score {XGBR_best_model.score(X_test, y_test)}")

Best Score: -155939.17856464014
Best Params: {'regressor__subsample': 0.7, 'regressor__n_estimators': 500, 'regressor__max_depth': 6, 'regressor__learning_rate': 0.1, 'regressor__colsample_bytree': 0.8}
Best Model Train Score 0.97699993001942
Best Model Test Score 0.7621002488879319


In [14]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# 1. THE OUTLIER CHOP (The Business Rule)
# We keep only properties under 1.5 Million Euros
df_filtered = df[df['price_eur'] <= 1500000].copy()

# 2. Re-define X and y on the clean dataset
cols_to_drop = ['locality', 'subtype', 'type_of_sale', 'price_eur']
X_clean = df_filtered.drop(columns=cols_to_drop)
y_clean = df_filtered['price_eur']

# 3. Apply the categorical trick to the new X_clean
for col in cat_cols:
    X_clean[col] = X_clean[col].astype('category')

# 4. Train/Test Split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

# 5. The Rescue Preprocessor and Model
rescue_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor), # (Your existing preprocessor for the ordinal building state)
    ('regressor', xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.1,          # 10x faster than the Grid Search winner!
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror', 
        tree_method='hist',           
        enable_categorical=True,      
        random_state=42
    ))
])

# 6. Fit and Grade
print("Running the Rescue Model...")
rescue_pipe.fit(X_train_c, y_train_c)

y_pred_train = rescue_pipe.predict(X_train_c)
y_pred_test = rescue_pipe.predict(X_test_c)

print(f"\n--- Rescue Model Results ---")
print(f"Train R2: {r2_score(y_train_c, y_pred_train):.4f}")
print(f"Test R2: {r2_score(y_test_c, y_pred_test):.4f}")

# The ultimate human-readable metric:
mae = mean_absolute_error(y_test_c, y_pred_test)
print(f"Mean Absolute Error (MAE): €{mae:,.2f}")

Running the Rescue Model...

--- Rescue Model Results ---
Train R2: 0.9607
Test R2: 0.7679
Mean Absolute Error (MAE): €66,997.57
